# memlayers on Colab: one run

Runs one experiment of the memory-layers project on a Colab GPU, with the run
dir and the 1.1 GB data bundle on Google Drive, so a disconnect loses at most
`runtime.ckpt_minutes` (15) of training: **re-running the run cell resumes**
(after a reconnect, too: `runtime=colab` sets `resume_foreign=true`, because a
new Colab VM has a new hostname).

1. Set the variables in the next cell. The default, `EXPERIMENT="smoke"`, is a
   one-minute check of the GPU install and the run layer that needs no data
   download; switch to `pkm` (etc.) once it passes.
2. Runtime -> Change runtime type -> a GPU (T4 runs fp16 + GradScaler; L4/A100 run bf16).
3. Run all cells. Optional W&B: add `WANDB_API_KEY` in Colab's **Secrets** panel
   (the key icon on the left) -- never paste a key into this notebook.

Phase-1 rule: every run of one comparison group (pkm seeds, pkm_randkey, dense)
on the same device type and precision. Do not mix Colab and Mac runs in a group.

In [ ]:
# ---- what to run --------------------------------------------------------
REPO_URL = "https://github.com/RoieGilad7/memory-layers-analysis.git"
REPO_COMMIT = "train/run-layer"   # a commit sha (preferred) or a branch
EXPERIMENT = "smoke"              # start here: ~1 min, no data download. Then pkm | pkm_randkey | dense | peer
SEED = 0
TAG = ""                          # e.g. "colab-pilot"; empty for a Phase-1 run
EXTRA = []                        # more overrides, e.g. ["train.steps=200"]
USE_WANDB = False                 # True: logging=wandb (online if the secret exists)

DRIVE = "/content/drive/MyDrive/memlayers"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess
os.makedirs(f"{DRIVE}/data", exist_ok=True)
os.makedirs(f"{DRIVE}/runs", exist_ok=True)

In [ ]:
# clone at the pinned commit (the repo itself is small; data and runs live on Drive)
%cd /content
!rm -rf memory-layers-analysis
!git clone -q {REPO_URL} memory-layers-analysis
%cd /content/memory-layers-analysis
!git checkout -q {REPO_COMMIT} && git log --oneline -1

In [ ]:
# the pinned environment, with the CUDA build of the pinned torch
!pip install -q torch==2.14.1 --index-url https://download.pytorch.org/whl/cu126
!pip install -q -r requirements.txt
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print("compute capability", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)

In [ ]:
# data v1 onto Drive (skips files that already verify), then the fixed eval sets.
# The smoke experiments read the 1% sample committed in the repo, so they skip this.
import os, subprocess
if EXPERIMENT.startswith("smoke"):
    print("smoke run: no data download needed (data/v1_sample is in the repo)")
else:
    subprocess.run(["python", "-m", "data.fetch", "--version", "v1", "--data-root", f"{DRIVE}/data"], check=True)
    sets = [f"{DRIVE}/data/v1/eval_sets/{n}.json" for n in ("heldout_sample", "fact_sample", "readlog_tokens")]
    # missing -> build them (deterministic: they must reproduce the committed hashes);
    # present -> check them; a mismatch stops here instead of being rebuilt over
    args = [] if all(os.path.exists(f) for f in sets) else None
    cmd = ["python", "-m", "memlayers.eval.eval_sets", "--version", "v1", "--data-root", f"{DRIVE}/data"]
    subprocess.run(cmd + (["--check"] if args is not None else []), check=True)


In [ ]:
# optional: W&B login from Colab's secrets panel (no key in this notebook)
logging_args = ["logging=files"]
if USE_WANDB:
    from google.colab import userdata
    try:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
        logging_args = ["logging=wandb", "logging.mode=online"]
    except Exception as e:  # no secret: mirror offline, backfill later with memlayers.wandb_sync
        print("no WANDB_API_KEY secret:", e)
        logging_args = ["logging=wandb", "logging.mode=offline"]

In [ ]:
# the run. Re-running this cell after a disconnect resumes from last.pt.
args = [f"experiment={EXPERIMENT}", f"seed={SEED}", "runtime=colab", *logging_args, *EXTRA]
if TAG:
    args.append(f"tag={TAG}")
print("python -m memlayers.run", " ".join(args))
!python -m memlayers.run {" ".join(args)}

In [ ]:
# compare the run this notebook just made (its own tables + plots on Drive).
# Comparing the whole runs folder mixes smoke runs (the 1% sample) with real-data
# runs, which compare refuses; for a cross-run table pass several run dirs.
RUN_ID = f"{EXPERIMENT}-s{SEED}" + (f"-{TAG}" if TAG else "")
!python -m memlayers.compare {DRIVE}/runs/{RUN_ID} --out {DRIVE}/compare/{RUN_ID}
!head -3 {DRIVE}/compare/{RUN_ID}/runs.csv
